[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_2_3/02_tag_2_3_gradient_descent_visualisierung.ipynb)

# Tag 2/3 - 02 Gradient Descent visualisieren

Gradient Descent passt Parameter in Richtung des negativen Gradienten an:

`neuer Parameter = alter Parameter - Lernrate * Gradient`

Das Beispiel nutzt lineare Regression mit nur einem trainierbaren Parameter `m`. Dadurch kann man die Loss-Funktion direkt als Kurve zeichnen.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
print("Setup abgeschlossen.")


In [ ]:
x = np.linspace(-2.5, 2.5, 80)
true_m = 1.8
fixed_b = 0.35
y = true_m * x + fixed_b + rng.normal(0, 0.45, size=len(x))


def predict(m):
    return m * x + fixed_b


def mse_loss(m):
    errors = predict(m) - y
    return np.mean(errors ** 2)


def mse_gradient(m):
    return np.mean(2 * (predict(m) - y) * x)


def gradient_descent_path(start_m=-3.0, learning_rate=0.12, steps=25):
    m = start_m
    path = []
    for step in range(steps + 1):
        path.append((step, m, mse_loss(m)))
        m = m - learning_rate * mse_gradient(m)
        if abs(m) > 20:
            break
    return path


m_grid = np.linspace(-4.0, 5.0, 300)
best_m = m_grid[np.argmin([mse_loss(m) for m in m_grid])]
print(f"Bestes m auf dem Grid: {best_m:.2f}; echte Steigung im Datensatz: {true_m:.2f}")


## Lernrate interaktiv ändern

Eine zu kleine Lernrate macht viele kleine Schritte. Eine zu große Lernrate kann über das Minimum springen oder sogar divergieren.


In [ ]:
def gd_demo(learning_rate=0.12, start_m=-3.0, steps=25):
    path = gradient_descent_path(start_m=start_m, learning_rate=learning_rate, steps=steps)
    path_m = np.array([row[1] for row in path])
    path_loss = np.array([row[2] for row in path])

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(m_grid, [mse_loss(m) for m in m_grid], color="black")
    axes[0].plot(path_m, path_loss, "o-", color="tab:orange")
    axes[0].scatter([best_m], [mse_loss(best_m)], color="green", s=90, label="ungefähres Minimum")
    axes[0].set_xlabel("Parameter m")
    axes[0].set_ylabel("MSE Loss")
    axes[0].set_title("Abstieg auf der Loss-Funktion")
    axes[0].legend()

    axes[1].scatter(x, y, alpha=0.7, label="Daten")
    axes[1].plot(x, predict(path_m[0]), "--", label=f"Start m={path_m[0]:.2f}")
    axes[1].plot(x, predict(path_m[-1]), color="tab:orange", linewidth=2, label=f"Ende m={path_m[-1]:.2f}")
    axes[1].plot(x, true_m * x + fixed_b, color="green", linewidth=2, alpha=0.8, label="wahre Funktion")
    axes[1].set_title(f"Modell nach {len(path)-1} Updates")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print(f"Letzter Parameter: m={path_m[-1]:.4f}")
    print(f"Letzter Loss: {path_loss[-1]:.4f}")


widgets.interact(
    gd_demo,
    learning_rate=widgets.FloatSlider(value=0.12, min=0.001, max=0.9, step=0.001, readout_format=".3f", description="Lernrate"),
    start_m=widgets.FloatSlider(value=-3.0, min=-4.0, max=5.0, step=0.1, description="Start m"),
    steps=widgets.IntSlider(value=25, min=1, max=80, step=1, description="Updates"),
);
